# ORCA-X — Colab GPU ML / Refinement Runner

This notebook is the single entry point for heavy XGBoost training/refinement work. It uses the repository's existing refinement implementations and changes only the compute backend.

**Runtime:** Runtime → Change runtime type → GPU → T4/L4.

In [ ]:
REPO_URL = 'https://github.com/Sayan260106/HackHeritage.git'
REPO_REF = 'feat/colab-refinement-gpu'  # Use main after this branch is merged.
REPO_DIR = '/content/HackHeritage'
import os
os.environ['ORCA_X_DEVICE'] = 'cuda'
os.environ['ORCA_X_N_JOBS'] = '2'

In [ ]:
!rm -rf "$REPO_DIR"
!git clone --depth 1 --branch "$REPO_REF" "$REPO_URL" "$REPO_DIR"
%cd "$REPO_DIR"
!python -m pip install -q --upgrade pip
!python -m pip install -q -r ml/requirements-colab.txt

In [ ]:
# Stop early rather than wasting a long run if the Colab GPU is unavailable.
!python ml/src/colab_preflight.py

## Canonical production training

Use this for the production ORCA-X XGBoost model. The canonical training script retains its forward +6h target, point-in-time feature contract, temporal validation and Digha spatial holdout.

In [ ]:
!python ml/src/train.py

## Heavy refinements

Run one refinement at a time. The compatibility runner covers the repository's XGBoost estimator classes without rewriting refinement logic.

In [ ]:
# Refinement 26 — uncertainty-aware forecast
!python ml/src/colab_gpu_runner.py ml/src/refinement26_uncertainty_aware_forecast.py

In [ ]:
# Refinement 25 — temporal reliability forecast
# !python ml/src/colab_gpu_runner.py ml/src/refinement25_temporal_reliability_forecast.py

## Other XGBoost scripts

Any existing script under `ml/src` that constructs `XGBClassifier`, `XGBRegressor` or `XGBRanker` can be launched through the same runner:

`!python ml/src/colab_gpu_runner.py ml/src/<script>.py`

This includes historical training, tuning, benchmarking and refinement scripts. Scripts that only perform pandas/NumPy/scikit-learn evaluation do not gain meaningful GPU acceleration and should simply run normally in Colab.